<a href="https://colab.research.google.com/github/aounraza379/SecureMed-Fed/blob/main/securemed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random

# 1. Set the number of data points (simulating about 15-16 minutes of data)
data_points = 1000

# 2. "Normal" heart rates (60 to 90 beats per minute)
heart_rates = [random.randint(60, 90) for _ in range(data_points)]

# 3. Create a "Cardiac Anomaly" (Emergency)
# Change the last 50 readings to be dangerously high
for i in range(950, 1000):
    heart_rates[i] = random.randint(150, 180)

# 4. Timestamp for each reading
timestamps = pd.date_range(start="2026-04-01 08:00:00", periods=data_points, freq='S')

# 5. Spreadsheet (DataFrame)
df = pd.DataFrame({
    'Timestamp': timestamps,
    'Heart_Rate': heart_rates
})

# 6. Save it as a CSV file
df.to_csv('patient_data.csv', index=False)

print("File 'patient_data.csv' has been created.")
print(df.tail(10))

In [ ]:
# 1. Load csv
data = pd.read_csv('patient_data.csv')

# 2. Safety Threshold (120 BPM is the danger zone for an elderly person at rest)
THRESHOLD = 120
alert_triggered = False

print("--- SecureMed Watchdog Scanning Started ---")

# 3. Scan 'Heart_Rate' column
for index, row in data.iterrows():
    current_bpm = row['Heart_Rate']
    timestamp = row['Timestamp']

    if current_bpm > THRESHOLD:
        print(f"ALERT! High Heart Rate Detected: {current_bpm} BPM at {timestamp}")
        alert_triggered = True

if alert_triggered:
    print("\n--- FINAL STATUS: CARDIAC ANOMALY DETECTED. EMERGENCY SERVICES NOTIFIED. ---")
else:
    print("\n--- FINAL STATUS: Patient is Stable. ---")

In [ ]:
import numpy as np
import hashlib

def generate_secure_update(heart_rate_column):
    """
    Simulates a Local Differential Privacy update for Federated Learning.
    Masks raw data before transmission to the central server.
    """
    # Calculate local statistics
    local_mean = heart_rate_column.mean()
    local_std = heart_rate_column.std()

    # Add Gaussian Noise (Differential Privacy) to mask the exact mean
    noise = np.random.normal(0, 0.1, 1)[0]
    masked_update = local_mean + noise

    # Generate a unique hash for the device to ensure data integrity (ZKP Foundation)
    device_id = "Device_Elderly_001"
    data_hash = hashlib.sha256(str(masked_update).encode()).hexdigest()

    return {
        "masked_mean": round(masked_update, 2),
        "integrity_hash": data_hash,
        "status": "Anomaly_Detected" if local_mean > 110 else "Normal"
    }

# Execute Local Privacy Masking
local_data = pd.read_csv('patient_data.csv')
secure_packet = generate_secure_update(local_data['Heart_Rate'])

print(f"--- Transmission Packet Created ---")
print(f"Masked Mean (Shared): {secure_packet['masked_mean']}")
print(f"Integrity Hash (Encrypted): {secure_packet['integrity_hash']}")
print(f"System Status: {secure_packet['status']}")